# 分布式光伏功率预测模型的稳健性测试与分层评估 · 教学参考

这是帮助理解写法的参考实现，不是官方标准答案。请在 `answer.ipynb` 独立作答；不要在本文件中覆盖自己的草稿。

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

data = pd.read_csv("solar_pv.csv").dropna().drop_duplicates()
print("清洗后形状：", data.shape)
X = data[["辐照度", "环境温度", "板温", "风速", "湿度", "云量", "装机容量", "安装倾角", "时刻"]]
y = data["是否高出力"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2026, stratify=y)
model = RandomForestClassifier(n_estimators=100, random_state=2026)
model.fit(X_train, y_train)
pred = model.predict(X_test)
baseline = accuracy_score(y_test, pred)
np.random.seed(2026)
perturbed = X_test + np.random.normal(0, 10, X_test.shape)
noise_accuracy = accuracy_score(y_test, model.predict(perturbed))
print("基线准确率：", baseline)
print("扰动后准确率：", noise_accuracy)
print("下降百分点：", (baseline - noise_accuracy) * 100)
hours = X_test["时刻"]
masks = {"早晨": hours.between(6, 9), "中午": hours.between(10, 14),
         "傍晚": hours.between(15, 18), "夜晚": (hours >= 19) | (hours <= 5)}
scores = {}
for period, mask in masks.items():
    if mask.any():
        scores[period] = accuracy_score(y_test[mask], pred[mask])
        print(period, scores[period])
    else:
        print(period, "无测试样本")
worst = min(scores, key=scores.get)
print("最低准确率时段：", worst, scores[worst])
